# Welcome!

Welcome to the first Jupyter notebook in the AI-orchestrated self-driving labs (47332) course. 
We will be using the Jupyter notebooks throughout the course. 

Since you are reading this, you have (probably) have managed to 

1.  install `PumpController` and `Odyssey` packages 
2.  download the notebooks for the course
3.  receive your pumpbot.

You will get some basic information about course exercises in this notebook (consider it a teaser) with significantly 
more to follow next week. 

**The main purpose of this notebook is to verify that you can use the course software and that you can communicate 
with the robot.**

## Exercise 0.1: Verify access to the PumpController

We will now test to see if you have the `PumpController` and can establish a connection with your pumpbot. 

Import the necessary modules for the `PumpController`

In [1]:
from pump_controller import PumpController, get_serial_port, list_serial_ports
from pump_controller import visualize_rgb, visualize_candidates
import time
import numpy as np

from pump_controller.mqtt_utils import request_task


In [8]:

task = '{"Mix": 12, "duration": 1}'


request_task(task)



Sending request with request id: aa1b1f33-92cd-4932-9670-ac074fd2da04
Timeout waiting for response.


If you run the command above without any error messages, it means that you have successfully installed the `PumpController`.

## Exercise 0.2.1: (USB): Establish connection to your pumpbot via USB serial (SKIP THIS IF YOU WANT ACCESS OVER WIFI)

Now we initialize the calibratebot. 

We need to figure out which port that the controller is connected to your computer. The `get_serial_port()` function should automatically do this for you, but things have a way of failing when you need them not to. That is why you can call the `list_serial_ports()` function to see all the ports on your computer, and you can simply use the correct port as a string input instead of using the `get_serial_port` function, if the command below does not work. 

The `cell_volume` and `drain_time` properties are already defined to 20 mL and 20 seconds, respectively, but you have the option of changing them here. Notice that a folder called *logs* is created with a file with the current timestamp in it - this is where the colors that you mix on this controller in this session will be stored.


We start with the `config_template.json` file as the `config_file` to begin with, but we will make our own in this tutorial.

Let's run the initialization command below.

In [ ]:
calibratebot = PumpController(ser_port = get_serial_port(), cell_volume = 20, drain_time = 20, config_file = 'config_files/config.json')

Exception: ERROR: No USB Serial Port Found. Please try again or define port manually using list_serial_ports().

If you have successfully initialized the pumpbot, you will see the "Arduino is ready" message. If the "Arduino is ready" message doesn't show up immediately, press the reset button on the Arduino.

Do you see the message? If so, you are now ready to move on to the next step.


## Exercise 0.2.2: (MQTT wireless): Establish connection to your pumpbot HiveMQTT broker

For connecting to the colorbot wirelessly from any wifi network, we use a the messaging protocol MQTT, and an MQTT broker to handle the messages. 

In [ ]:
calibratebot = PumpController(ser_port = None, 
                              cell_volume = 20, 
                              drain_time = 20, 
                              config_file = "config_files/config.json")






Computer client connected to MQTT broker.
.....................................................


[DISPATCH] Got message on colorbot-4/data: b'{"req_id":"9a5f1c10-f4e5-4fa5-87d9-d4424e7b32e8","color_sense":{"r":70,"g":103,"b":94}}'
Handlers for this topic: None
[DISPATCH] Got message on machines/colorbot-4/status: b'offline'
Handlers for this topic: None


## MQTT communication 
Communicating over MQTT is done via topics, that clients can pupblish and sbscirbe to. The colorbot subscribes to the **/command** topic, and sends data return on the /data topic. The computer clients does the opposite. The data send is formated as json.
Now we can try sending messages on the MQTT topic /command. 
- Send a command on the topic /command:

In [184]:
topic_name = "colorbot-4/command"

color_white = {"r": 255, "g": 145, "b": 36}
color_capex_green = {"r": 20, "g": 255, "b": 40}

calibratebot.mqtt.publish(topic_name, {"led_color": color_capex_green})



Send and recieve

In [185]:
rgb = calibratebot.measure(timeout=12.0)
print("Measured", rgb)



Connecting to broker with result code 0
Sending measurement request with request id: b63e5da9-786a-465e-ad82-1fa9f235959f
Received RGB: [2, 5, 9]
Measured [2, 5, 9]


In [6]:
import time, datetime
import paho.mqtt.client as mqttClient
Connected = False
# using public available service from hivemq
broker_address="broker.hivemq.com"
port = 1883
user = "roger" # default
password = "password" # default

In [7]:

def on_log(client, userdata, level, buf):
 print( str(datetime.datetime.now()) + ": ",buf)

def on_message(client, userdata, message):
 print ("Client received: " + str(message.topic))
 print ("Message received: " + str(message.payload))

def on_connect(client, userdata, flags, rc):
 if rc == 0:
  print("Connected to broker")
  global Connected
  Connected = True
 else:
  print("Connection failed")

In [9]:
client = mqttClient.Client(client_id = "someDeviceChannel") #create new instance


# datascience will just subscribe in this example, but can also publish!
client.on_connect = on_connect #attach function to callback
client.on_message = on_message #attach function to callback
client.on_log = on_log #attach function to callback
client.username_pw_set(user, password=password)
client.connect(broker_address, port=port)

2025-05-24 14:55:20.162470:  Sending CONNECT (u1, p1, wr0, wq0, wf0, c1, k60) client_id=b'someDeviceChannel'


/var/folders/fr/s1wl518j42d0chw5112zh11h0000z9/T/ipykernel_36286/1886457661.py:1: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqttClient.Client(client_id = "someDeviceChannel") #create new instance


<MQTTErrorCode.MQTT_ERR_SUCCESS: 0>

In [10]:
client.loop_start()

<MQTTErrorCode.MQTT_ERR_SUCCESS: 0>

2025-05-24 14:55:23.050067:  Received CONNACK (0, 0)
Connected to broker


2025-05-24 14:55:25.511260:  Received SUBACK
2025-05-24 14:55:25.529016:  Received SUBACK


In [13]:

while Connected != True:
  time.sleep(0.1)
client.subscribe("someDeviceChannel/photocell")
client.subscribe("someDeviceChannel/coll")
#client.subscribe("otherchannel/onoff")

try:
 while True:
  time.sleep(1)
except KeyboardInterrupt:
 client.disconnect()
 client.loop_stop()

In [19]:
import paho.mqtt.client as mqtt
import secrets_mqtt as mqtt_conf
import time
import ssl

def on_connect(client, userdata, flags, rc):
    print("Connected with result code", rc)
    client.subscribe("colorbot-4/data")

def on_message(client, userdata, msg):
    print("Topic:", msg.topic, "Payload:", msg.payload)

client = mqtt.Client()
client.username_pw_set(mqtt_conf.USERNAME, mqtt_conf.PASSWORD)
client.on_connect = on_connect
client.on_message = on_message
client.tls_set(cert_reqs=ssl.CERT_NONE)  # if using TLS, match ESP settings
client.tls_insecure_set(True)

client.connect(mqtt_conf.BROKER_HOST, mqtt_conf.PORT)
client.loop_start()

# Let the loop run, so messages can arrive
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    client.disconnect()
    client.loop_stop()


/var/folders/fr/s1wl518j42d0chw5112zh11h0000z9/T/ipykernel_36286/2201861666.py:13: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client()


Connected with result code 0
Topic: colorbot-4/data Payload: b'{"req_id":"9a5f1c10-f4e5-4fa5-87d9-d4424e7b32e8","color_sense":{"r":70,"g":103,"b":94}}'


## Exercise 0.3: Controlling the pumps of your pumpbot

For calibrating the pumps, we will not be using the test cell, but running the liquid from the respective bottles into a bottle on the scale. So removing the hoses from the test cell and place such that they dispense onto a bottle on the weighing scale.

Before doing anything else, we should purge all the pumps. This means filling all of the tubes with their respective liquids. We do this by using the `purge_pump` function, which takes the pump name ('R', 'G', 'B', 'Y', 'W', 'D') and the time (in seconds) to run the pump, as variables. 

As you can probably guess, the names correspond to the following:

* 'R': red
* 'G': green
* 'B': blue
* 'Y': yellow
* 'W': water
* 'D': drain
   
Do this one-by-one until all tubes are filled with liquid. The drain tubes of course do not need to be purged. 

After purging the red, green, blue, yellow and water pumps, empty the weighing bottle.

In [3]:
# Water
#calibratebot.purge_pump('W', 1)


# RGBY
calibratebot.purge_pump('R', 1)

# Pump G not connected
#calibratebot.purge_pump('G', 3)

#calibratebot.purge_pump('B', 1)

# Pump Y (werid somhow)
#calibratebot.purge_pump('Y', 3)

#Drain:# 
#calibratebot.drain()

# Drain for custom time (if required):
#calibratebot.drain(drain_time = 10)

# Flush:
# calibratebot.flush()

#for i in range(10):
#    calibratebot.purge_pump('B', 0.05)
#    time.sleep(1)

Purging pump R for 1 seconds...


In [182]:
calibratebot.drain(drain_time = 10)

Purging pump D for 10 seconds...


In [13]:
# For measurement
reply = request_task("Meas")
print("Measurement reply:", reply)

# For mixing
#reply = request_task({"Mix": 26, "duration": 15})
#print("Mix reply:", reply)

Sending request with request id: e0110fd0-6d28-4805-b6e8-d8582926ad29
Measurement reply: {'req_id': 'e0110fd0-6d28-4805-b6e8-d8582926ad29', 'color_sense': {'r': 2, 'g': 5, 'b': 9}}


## Exercise 0.4: Calibrating the pumps

Now we want to calibrate the pumps. We need to figure out how many mL of water comes out of the pump for every second that we run it. We do this by running the pump for certains amounts of times and weigh the amount of liquid that is output. We can then do a linear regression in Excel to find the slope and the offset for each pump. As each pump is different, these values will also be different. 


The `calibration_test` function cycles through different amounts of time of running the pumps, and waits for 5 seconds. In these 5 seconds, your job is to read the measurement on the weighing scale and note it down in the Excel sheet.

In [126]:
times = [0.1, 0.2, 0.4, 0.8, 1.0, 1.6, 2.0, 2.5, 3.0, 5.0, 7.0, 8.0]

def calibration_test(pump, times):
    for t in times:
        calibratebot.purge_pump(pump, t)
        print(f"sleeping after {t} on {pump}")
        print("-----------")
        print()
        
        time.sleep(5.0)

Let us run the calibration now! 

Open up the `calibrate_files/pump_calibration_template.xlsx` file and get ready to note down the weights. Optimally, you should do three calibrations for each pump - this results in 6 x 3 = 18 calibrations. If you are pressed for time, just do one calibration per pump and copy the results to the other columns for the same pump. 

Finally, you can take the average `a` (slope) and `b` (intercept) values on the `Main` page of the Excel sheet and write them into a `config.json file`. A template for this is given in `config_files/config_template.json`. Do not change the `pin` values for the different pumps!

In [146]:
times = np.linspace(0.0, 0.2, 11)

calibration_test('W', times)

# calibration_test('G', times)
# calibration_test('B', times)
# calibration_test('Y', times)
# calibration_test('W', times)
# calibration_test('D', times)

Purging pump W for 0.0 seconds...
sleeping after 0.0 on W
-----------

Purging pump W for 0.02 seconds...
sleeping after 0.02 on W
-----------

Purging pump W for 0.04 seconds...
sleeping after 0.04 on W
-----------

Purging pump W for 0.06 seconds...
sleeping after 0.06 on W
-----------

Purging pump W for 0.08 seconds...
sleeping after 0.08 on W
-----------

Purging pump W for 0.1 seconds...
sleeping after 0.1 on W
-----------

Purging pump W for 0.12 seconds...
sleeping after 0.12 on W
-----------

Purging pump W for 0.14 seconds...
sleeping after 0.14 on W
-----------

Purging pump W for 0.16 seconds...
sleeping after 0.16 on W
-----------

Purging pump W for 0.18 seconds...
sleeping after 0.18 on W
-----------

Purging pump W for 0.2 seconds...
sleeping after 0.2 on W
-----------



Take these calibration slope and intercept values for each of the pumps, and note them down in the `config.json` file